# Simple Blockchain in Python
A minimal blockchain that links blocks using SHA-256 hashes and verifies chain integrity.
Run all cells in order (Runtime > Run all).

In [1]:
import hashlib
import json
import time

In [2]:
class Block:
    def __init__(self, index, data, previous_hash, timestamp=None, nonce=0):
        self.index = index
        self.timestamp = timestamp or time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.compute_hash()

    def compute_hash(self):
        """Create a SHA-256 hash of the block's contents."""
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def __repr__(self):
        return (f"Block #{self.index}\n"
                f"  Timestamp     : {self.timestamp}\n"
                f"  Data          : {self.data}\n"
                f"  Previous Hash : {self.previous_hash}\n"
                f"  Hash          : {self.hash}\n"
                f"  Nonce         : {self.nonce}\n")

In [3]:
class Blockchain:
    def __init__(self, difficulty=2):
        self.difficulty = difficulty          # number of leading zeros required (simple PoW)
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        return Block(0, "Genesis Block", "0")

    def get_latest_block(self):
        return self.chain[-1]

    def proof_of_work(self, block):
        """Very simple proof-of-work: find a nonce so hash starts with N zeros."""
        target = "0" * self.difficulty
        while not block.hash.startswith(target):
            block.nonce += 1
            block.hash = block.compute_hash()
        return block

    def add_block(self, data):
        previous_block = self.get_latest_block()
        new_block = Block(
            index=previous_block.index + 1,
            data=data,
            previous_hash=previous_block.hash
        )
        new_block = self.proof_of_work(new_block)
        self.chain.append(new_block)
        return new_block

    def is_chain_valid(self):
        """Verify integrity: recompute hashes and check the links."""
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            # 1. Has this block's own data been tampered with?
            if current.hash != current.compute_hash():
                print(f"Block {current.index} has been tampered with (hash mismatch).")
                return False

            # 2. Does it correctly point to the previous block?
            if current.previous_hash != previous.hash:
                print(f"Block {current.index} is not linked correctly to Block {previous.index}.")
                return False

            # 3. Does it still satisfy the proof-of-work difficulty?
            if not current.hash.startswith("0" * self.difficulty):
                print(f"Block {current.index} does not satisfy proof-of-work difficulty.")
                return False

        return True

In [4]:
def print_chain(blockchain):
    for block in blockchain.chain:
        print(block)

## Build the chain and print its contents

In [5]:
my_chain = Blockchain(difficulty=3)
my_chain.add_block("Alice pays Bob 10 coins")
my_chain.add_block("Bob pays Charlie 5 coins")
my_chain.add_block("Charlie pays Dave 2 coins")

print("===== Blockchain Contents =====\n")
print_chain(my_chain)

===== Blockchain Contents =====

Block #0
  Timestamp     : 1785300430.1100118
  Data          : Genesis Block
  Previous Hash : 0
  Hash          : 4de7f29f1654c4390f6be2fab0bc6a8c5d786ff21b0c09378a2d07894aa181dc
  Nonce         : 0

Block #1
  Timestamp     : 1785300430.1100726
  Data          : Alice pays Bob 10 coins
  Previous Hash : 4de7f29f1654c4390f6be2fab0bc6a8c5d786ff21b0c09378a2d07894aa181dc
  Hash          : 0006777cd920d222a5b40d7ce959739e26aeea6f5599dee388da3b75a0f1a91e
  Nonce         : 4175

Block #2
  Timestamp     : 1785300430.1281602
  Data          : Bob pays Charlie 5 coins
  Previous Hash : 0006777cd920d222a5b40d7ce959739e26aeea6f5599dee388da3b75a0f1a91e
  Hash          : 00071fbf45ddb3159a30b769475cb558a90e22f700276a2c65aad3df7c54bb30
  Nonce         : 9460

Block #3
  Timestamp     : 1785300430.1680207
  Data          : Charlie pays Dave 2 coins
  Previous Hash : 00071fbf45ddb3159a30b769475cb558a90e22f700276a2c65aad3df7c54bb30
  Hash          : 000cfe31f9b5ee789

## Check integrity before tampering

In [6]:
print("Chain valid?", my_chain.is_chain_valid())

Chain valid? True


## Tamper with a block and check again

In [7]:
print("Tampering with Block 1 data...\n")
my_chain.chain[1].data = "Alice pays Bob 1000 coins"
# Note: hash is NOT recomputed here, simulating a naive attacker
# who edits data but doesn't fix the hash — is_chain_valid() will catch it.

print("Chain valid?", my_chain.is_chain_valid())

Tampering with Block 1 data...

Block 1 has been tampered with (hash mismatch).
Chain valid? False
